# K-Dimensional Pdits

In [1]:
import equinox as eqx
import jax
import jax.numpy as jnp
from torx import (
    CompiledSamplePCircuit,
    DiscretePCircuit,
    draw_circuit,
    PditCycle,
    PditShift,
    PditSWAP,
    PNOT,
    SampleSimulator,
    StateVectorSimulator,
)

In [2]:
gate = PditShift(jnp.inf, sites=0, dims=3)

print("PditShift transition matrix (p=1):")
print(gate.get_matrix())

PditShift transition matrix (p=1):
[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [3]:
circuit = DiscretePCircuit([PditShift(jnp.inf, sites=0, dims=3)])
sim = StateVectorSimulator()
sim_fn = eqx.filter_jit(sim.density)

# Start in state |0)
x = jnp.array([1.0, 0.0, 0.0])
print(f"Initial: {x}")

for i in range(4):
    x = sim_fn(circuit, x)
    print(f"After shift {i + 1}: {x}")

Initial: [1. 0. 0.]
After shift 1: [0. 1. 0.]
After shift 2: [0. 0. 1.]
After shift 3: [1. 0. 0.]
After shift 4: [0. 1. 0.]


In [4]:
circuit = DiscretePCircuit(
    [
        PNOT(0.0, 0),
        PditShift(0.0, sites=1, dims=3),
    ]
)

print(f"Circuit dimensions: {circuit.dims}")
print(f"State space size: {2 * 3} = 6")

Circuit dimensions: (2, 3)
State space size: 6 = 6


In [5]:
x = jnp.zeros(6).at[0].set(1.0)
result = sim_fn(circuit, x)

print("Initial state: |0,0)")
print(f"Final distribution: {result}")
for i in range(2):
    for j in range(3):
        idx = i * 3 + j
        print(f"  |{i},{j}) (idx={idx}): {result[idx]:.3f}")

Initial state: |0,0)
Final distribution: [0.25 0.25 0.   0.25 0.25 0.  ]
  |0,0) (idx=0): 0.250
  |0,1) (idx=1): 0.250
  |0,2) (idx=2): 0.000
  |1,0) (idx=3): 0.250
  |1,1) (idx=4): 0.250
  |1,2) (idx=5): 0.000


In [6]:
circuit = DiscretePCircuit([PditSWAP(jnp.inf, sites=[0, 1], dims=3)])

print(f"Circuit dimensions: {circuit.dims}")
print(f"State space size: {3 * 3} = 9")

# Start in |1,2) (index = 1*3 + 2 = 5)
x = jnp.zeros(9).at[5].set(1.0)
result = sim_fn(circuit, x)

print("\nInitial: |1,2)")
print("After SWAP: |2,1) (index = 2*3 + 1 = 7)")
print(f"Result: {result}")

Circuit dimensions: (3, 3)
State space size: 9 = 9

Initial: |1,2)
After SWAP: |2,1) (index = 2*3 + 1 = 7)
Result: [0. 0. 0. 0. 0. 0. 0. 1. 0.]


## Sample Simulator with Pdits

In [7]:
pdit_circuit = DiscretePCircuit([PditShift(0.5, sites=0, dims=4)])
sample_sim = SampleSimulator(num_samples=1000)
compiled = CompiledSamplePCircuit.from_pcircuit(pdit_circuit)

key = jax.random.key(42)
samples = sample_sim.sample(compiled, jnp.array([0]), key)

print("Starting state: 0")
print(f"Samples shape: {samples.shape}")
print(f"Unique values: {jnp.unique(samples)}")
print(f"{jax.nn.sigmoid(0.5):.3f}, {samples.mean():.3f}")

Starting state: 0
Samples shape: (1000, 1)
Unique values: [0 1]
0.622, 0.601


In [8]:
mixed_circuit = DiscretePCircuit(
    [
        PNOT(1.0, 0),
        PditCycle(jnp.array([1.0, 1.0]), 1, dims=3),
        PditSWAP(1.0, [1, 2], dims=3),
    ]
)
draw_circuit(mixed_circuit)
print(f"Circuit dimensions: {mixed_circuit.dims}")

mixed_compiled = CompiledSamplePCircuit.from_pcircuit(mixed_circuit)
samples = sample_sim.sample(mixed_compiled, jnp.array([0, 0, 0]), key)
print(f"Samples shape: {samples.shape}")
print(f"First 5 samples:\n{samples[:5]}")

       ┌──────┐                 
p_0: ──┤ PNOT ├─────────────────
       └──────┘                 
     ┌───────────┐┌──────────┐  
p_1: ┤ PditCycle ├┤ PditSWAP ├──
     └───────────┘│          │  
                  │          │  
p_2: ─────────────┤          ├──
                  └──────────┘  
Circuit dimensions: (2, 3, 3)
Samples shape: (1000, 3)
First 5 samples:
[[1 0 2]
 [1 0 0]
 [1 0 1]
 [0 0 0]
 [1 1 0]]


In [9]:
def sample_loss(compiled_circuit, key):
    return sample_sim.expval_all(compiled_circuit, jnp.array([0]), key).sum()


grad_fn = eqx.filter_grad(sample_loss)
grad = grad_fn(compiled, key)
print(f"Gradient of pdit circuit: {grad.thetas}")

Gradient of pdit circuit: [[0.23500371]]
